# Project II: Economic Growth 

In [ ]:
### A. Settings
%load_ext autoreload
%autoreload 2

# i. Importing packages
import pandas as pd 
import numpy as np 
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from numpy import linalg as la
from tabulate import tabulate
from scipy import stats
from pathlib import Path

# ii. Importing methods from python script
import Project2

# iii. Plotting settings
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.family':'serif','mathtext.fontset':'cm'})
plt.rcParams.update({'font.size': 12})


## Read data 

In [ ]:
dat = pd.read_csv('growth.csv')
lbldf = pd.read_csv('labels.csv', index_col='variable')
lbl_all = lbldf.label.to_dict() # as a dictionary
print(f'The data contains {dat.shape[0]} rows (countries) and {dat.shape[1]} columns (variables).')

In [ ]:
# convenient to keep a column of ones in the dataset
dat['constant'] = np.ones((dat.shape[0],))

# Collections of variables

In order to make the analysis simpler, it may be convenient to collect variables in sets that belong together naturally. 

In [ ]:
# all available variables
vv_institutions = ['marketref', 'dem', 'demCGV', 'demBMR', 'demreg'] 
vv_geography = [
        'tropicar','distr', 'distcr', 'distc','suitavg','temp', 'suitgini', 'elevavg', 'elevstd',
        'kgatr', 'precip', 'area', 'abslat', 'cenlong', 'area_ar', 'rough','landlock', 
        'africa',  'asia', 'oceania', 'americas' # 'europe' is the reference
]
vv_geneticdiversity = ['pdiv', 'pdiv_aa', 'pdivhmi', 'pdivhmi_aa']
vv_historical = ['pd1000', 'pd1500', 'pop1000', 'pop1500', 'ln_yst'] # these are often missing: ['pd1', 'pop1']
vv_religion = ['pprotest', 'pcatholic', 'pmuslim']
vv_danger = ['yellow', 'malfal',  'uvdamage']
vv_resources = ['oilres', 'goldm', 'iron', 'silv', 'zinc']
vv_educ = ['ls_bl', 'lh_bl'] # secondary, tertiary: we exclude 'lp_bl' (primary) to avoid rank failure 

vv_all = {'institutions': vv_institutions, 
          'geography': vv_geography, 
          'geneticdiversity': vv_geneticdiversity,
          'historical': vv_historical,
          'religion': vv_religion,
          'danger':vv_danger, 
          'resources':vv_resources
         }
list_of_lists = vv_all.values()
vv_all['all'] = [v for sublist in list_of_lists for v in sublist]


# Selection of controls

This section reduces the candidate controls before the post-double-selection lasso. No convergence regression is estimated here. Each step adds the variables it drops to `vv_dropped`; `dat` itself is not changed.

## 1. Load and inspect

`dat` and `lbl_all` are used as loaded above. The header of `growth.csv` lists `pd1500` twice, and pandas names the second copy `pd1500.1`. If the copy equals `pd1500`, it carries no extra information and is dropped.

In [ ]:
# 1. pandas reads the second pd1500 column as pd1500.1
print(f"pd1500 and pd1500.1 are equal: {dat['pd1500'].equals(dat['pd1500.1'])}")

# 2. drop the copy
vv_dropped = {'pd1500.1': 'load'}

## 2. Trim the sample

The convergence regression needs both `gdp_growth` and `lgdp_initial`, so all later steps use only the countries where both are observed (`dat_trim`).

In [ ]:
# 1. countries with both outcome and regressor
I = dat[['gdp_growth', 'lgdp_initial']].notnull().all(axis=1)

# 2. trimmed sample without the dropped copy
dat_trim = dat.loc[I, [v for v in dat.columns if v not in vv_dropped]]
print(f'Countries before trimming: {dat.shape[0]}, after trimming: {dat_trim.shape[0]}')

## 3. Solow core variables

One variable per Solow concept is protected: it is never dropped in the steps below. `lgdp_initial` is protected as the regressor of interest.

### Economic theory

[To be written by the group]

In [ ]:
# 1. one variable per Solow concept
vv_solow = {'Population growth': 'pop_growth',
            'Saving / interest rate': 'investment_rate',
            'Technology growth (proxy)': 'lh_bl',
            'Human capital': 'ls_bl',
            'Capital growth (proxy)': 'capital_growth_pct_gdp_initial'}

# 2. protected variables are never dropped
protected = ['lgdp_initial'] + list(vv_solow.values())

# 3. non-missing values in the trimmed sample
pd.DataFrame({'concept': list(vv_solow.keys()),
              'chosen': list(vv_solow.values()),
              'non-missing': dat_trim[list(vv_solow.values())].notnull().sum().values})

| Concept | Candidates in the data | Why the chosen variable |
|---|---|---|
| Population growth | `pop_growth`, `population_initial`, `population_now` | `pop_growth` is the growth rate itself; the two levels are only its start and end points. |
| Saving / interest rate | `investment_rate`, `ginv`, `capital_growth_pct_gdp_initial`, `capital_growth_pct_gdp_now` (no interest rate in the data) | `investment_rate` is the investment share of GDP averaged over the whole growth period, the measure named in the assignment. |
| Technology growth | `lh_bl`; institutions: `polity`, `polity2`, `dem`, `demBMR`, `demCGV`, `marketref`, `currentinst`, `legor_uk`, `legor_fr`; history: `ln_yst`, `pd1500` | No direct measure exists. `lh_bl` (proxy): a workforce with tertiary education is what develops and adopts new technology. |
| Human capital | `ls_bl`, `lp_bl`, `lh_bl`, `leb95`, `imr95` | `ls_bl`: secondary education is the broadest skill measure not already used as the technology proxy; `leb95` and `imr95` are health measures from 1995. |
| Capital growth | `capital_growth_pct_gdp_initial`, `capital_growth_pct_gdp_now`, `ginv`, `investment_rate` | No capital stock in the data. `capital_growth_pct_gdp_initial` (proxy) is, despite its name, investment in % of GDP in 1970: capital accumulation at the start of the period, which later growth cannot affect. |

## 4. Mechanical identities and leakage

Variables linked by definition carry the same information. The data come from different sources, so the relations are applied by definition and are not tested on the data. Rule: drop the non-protected variable with the most missing values in the trimmed sample. On a tie, drop the derived variable or, for a set that sums to a constant, the base category already used in the notebook. Variables that construct the outcome or the regressor are always dropped (leakage). Numbers in brackets are missing values in the trimmed sample.

In [ ]:
# 1. relations by definition: relation, linked variables, dropped variables
vv_rel = [
    ['leakage: construct the regressor', ['gdp_pc_initial', 'gdp_initial'], ['gdp_pc_initial', 'gdp_initial']],
    ['leakage: construct the outcome', ['gdp_pc_now', 'gdp_now'], ['gdp_pc_now', 'gdp_now']],
    ['lpop_initial = ln(population_initial)', ['lpop_initial', 'population_initial'], ['lpop_initial']],
    ['pop_growth from population_initial and population_now',
     ['pop_growth', 'population_initial', 'population_now'], ['population_now']],
    ['mortality, its log and exp(log)', ['mortality', 'logem4', 'imputedmort'], ['mortality', 'imputedmort']],
    ['pd1 = pop1 / area', ['pd1', 'pop1', 'area'], ['pd1']],
    ['pd1000 = pop1000 / area', ['pd1000', 'pop1000', 'area'], ['pd1000']],
    ['pd1500 = pop1500 / area', ['pd1500', 'pop1500', 'area'], ['pd1500']],
    ['continent dummies sum to 1', ['africa', 'asia', 'oceania', 'americas', 'europe'], ['europe']],
    ['religion shares sum to 100', ['pprotest', 'pcatholic', 'pmuslim', 'pother'], ['pother']],
    ['Barro-Lee education shares', ['lp_bl', 'ls_bl', 'lh_bl'], ['lp_bl']],
    ['polity2 = polity recoded', ['polity', 'polity2'], ['polity']],
    ['distcr from distc and distr', ['distcr', 'distc', 'distr'], ['distcr']],
]

# 2. add the dropped variables to vv_dropped
for rel, vv, drop in vv_rel:
    for v in drop:
        vv_dropped[v] = 'leakage' if rel.startswith('leakage') else 'identity'


# 3. assert that non-missing polity2 values are within the -10 to 10 scale
assert dat_trim['polity2'].dropna().between(-10, 10).all(), \
    "polity2 contains non-missing values outside the -10 to 10 scale"

# 4. table with missing values in the trimmed sample in brackets
n_miss = dat_trim.isnull().sum()
pd.DataFrame({
    'relation': [r[0] for r in vv_rel],
    'variables': [', '.join(f'{v} ({n_miss[v]})' for v in r[1]) for r in vv_rel],
    'dropped': [', '.join(r[2]) for r in vv_rel]
})

- **Leakage (regressor):** `gdp_pc_initial` is `lgdp_initial` before taking logs, and `gdp_initial` is `gdp_pc_initial` times population. Both are dropped.
- **Leakage (outcome):** `gdp_pc_now` and `gdp_pc_initial` together give `gdp_growth`, and `gdp_now` is `gdp_pc_now` times population. Both are dropped.
- **`lpop_initial` = ln(`population_initial`):** neither is missing, so the derived `lpop_initial` is dropped and population enters in levels.
- **`pop_growth`** is the annual growth rate from `population_initial` to `population_now`. `pop_growth` is protected and none of the three is missing. Given `population_initial` and `pop_growth`, `population_now` is the derived level and is dropped.
- **`mortality`, `logem4`, `imputedmort`** are one mortality measure in levels, logs and exp(log). `mortality` (51 missing) and `imputedmort` (35) are dropped, and `logem4` (31) is kept.
- **Population density = population / `area`:** in 1, 1000 and 1500 CE, density and population have the same number of missing values (17, 7 and 3), so the derived densities are dropped. `area` has none.
- **Continent dummies** sum to one and none is missing, so the base category `europe` (as in `vv_geography`) is dropped.
- **Religion shares** sum to 100 and none is missing, so the base category `pother` (as in `vv_religion`) is dropped.
- **Barro-Lee shares** split the population by education level. `ls_bl` and `lh_bl` are protected, so `lp_bl` (as in `vv_educ`) is dropped.
- **`polity2`** is `polity` recoded. Both have 16 missing values, but `polity2` corrects the invalid `-88` codes for PAK and SDN. Therefore, `polity` is dropped and `polity2` is retained for the analysis.
- **`distcr`** (distance to coast or river) is a function of `distc` and `distr`. All three have 9 missing, so the derived `distcr` is dropped.

## 5. Timing

The remaining candidates below are measured after 1970 according to their labels. Their values may respond to growth over the period itself (bad controls). They are kept for now, and the group decides:

- `investment_rate` (average 1970-2020) and `pop_growth` (1970-2020), both protected
- `capital_growth_pct_gdp_now` (2020)
- `currentinst` (ease of doing business, 2012)
- `imr95` and `leb95` (infant mortality and life expectancy, 1995)
- `yellow` (yellow fever vector present "today")

Many labels state no year (for example the Barro-Lee shares and the democracy measures), so their timing is unknown.

## 6. Correlation pruning

Two controls with an absolute correlation of 0.95 or more (rounded half up to two decimals) carry almost the same information. Pairs are taken from the highest correlation down. In each pair, the variable with fewer missing values is kept and protected variables are never dropped. Ties are decided by economic interpretation, and pairs that contain an already dropped variable are skipped. `lgdp_initial` is included so that no control can be a near-copy of the regressor. A tiny constant is added before rounding so that exact halves round up despite floating-point representation. Numbers in brackets are missing values in the trimmed sample.

In [ ]:
# 1. remaining candidates, including the regressor
vv_cand = [v for v in dat_trim.columns if v not in vv_dropped and v not in ['code', 'gdp_growth']]

# 2. absolute correlations above the diagonal, highest first
C = dat_trim[vv_cand].corr().abs()
upper = np.triu(np.ones(C.shape, dtype=bool), k=1)
pairs = C.where(upper).stack().sort_values(ascending=False)

# 3. round half up to two decimals and keep pairs of 0.95 or more
pairs = (pairs + 1e-9).round(2)
pairs = pairs[pairs >= 0.95]

# 4. drop the variable with more missing values; ties are decided by hand
vv_tie = {('demCGV', 'demBMR'): 'demCGV', ('dem', 'demBMR'): 'demBMR'}
rows = []
for (a, b), r in pairs.items():
    if a in vv_dropped or b in vv_dropped or (a in protected and b in protected):
        drop = ''
    elif a in protected or b in protected:
        drop = b if a in protected else a
    elif n_miss[a] != n_miss[b]:
        drop = a if n_miss[a] > n_miss[b] else b
    else:
        drop = vv_tie[(a, b)]
    if drop:
        vv_dropped[drop] = 'correlation'
    rows.append([f'{a} ({n_miss[a]})', f'{b} ({n_miss[b]})', r, drop])

pd.DataFrame(rows, columns=['a', 'b', 'corr', 'dropped'])

Drops and ties (missing values in brackets):

- `pop1000` (7) and `pop1` (17) are dropped against `pop1500` (3): population levels in different centuries are near-copies of each other. `pop1500` (3) is then dropped against `population_initial` (0), because in levels both are dominated by the same large countries.
- `pdivhmi` (24) and `pdivhmi_aa` (25) are dropped against `pdiv` (0) and `pdiv_aa` (5). They are alternative versions of predicted genetic diversity.
- `ginv` (16) is dropped against the protected `capital_growth_pct_gdp_initial`: both measure investment as a share of GDP.
- `abslat` (1) is dropped against `uvdamage` (0). UV exposure is largely determined by distance from the equator.
- **Tie, `demCGV` and `demBMR` (12 each).** Both are 0/1 codings of democracy, but `demCGV` also requires that power has changed hands, so a stable democracy where one party keeps winning is coded as non-democratic. `demBMR` is easier to interpret and is kept.
- **Tie, `dem` and `demBMR` (12 each).** `dem` is the source's own democracy measure (ANRR) and `demBMR` an alternative coding of the same concept, so `dem` is kept.
- The other three pairs involve a variable that was already dropped and need no action.

## 7. Missingness

The lasso needs every control observed for every country. Controls missing for more than `MAX_MISSING_SHARE` of the trimmed sample are dropped (protected controls never are), and then only complete cases are kept.

In [ ]:
# 1. share of missing values per remaining control in the trimmed sample
MAX_MISSING_SHARE = 0.20
vv_x = [v for v in dat_trim.columns if v not in vv_dropped and v not in ['code', 'gdp_growth', 'lgdp_initial']]
miss = dat_trim[vv_x].isnull().mean()

# 2. drop controls above the threshold, never protected ones
for v in vv_x:
    if miss[v] > MAX_MISSING_SHARE and v not in protected:
        vv_dropped[v] = 'missingness'
print(f"Dropped (missing share above {MAX_MISSING_SHARE}): {[v for v in vv_x if v in vv_dropped]}")

# 3. table of missing shares
pd.DataFrame({'missing share': miss,
              'protected': [v in protected for v in vv_x],
              'dropped': [v in vv_dropped for v in vv_x]}).sort_values('missing share', ascending=False).round(3)

In [ ]:
# 1. remaining controls
vv_x = [v for v in vv_x if v not in vv_dropped]

# 2. keep complete cases on outcome, regressor and controls
J = dat_trim[['gdp_growth', 'lgdp_initial'] + vv_x].notnull().all(axis=1)
dat_pre = dat_trim.loc[J, ['code', 'gdp_growth', 'lgdp_initial'] + vv_x]
print(f'Countries in the trimmed sample: {dat_trim.shape[0]}, complete cases: {dat_pre.shape[0]}')

In [ ]:
# 1. countries lost and the variables they miss
L = dat_trim.loc[~J, ['code'] + vv_x]
pd.DataFrame({'code': L['code'],
              'n missing': L[vv_x].isnull().sum(axis=1),
              'missing': L[vv_x].isnull().apply(lambda r: ', '.join(r.index[r]), axis=1)}).sort_values(['n missing', 'code'])

Nine controls are missing for more than `MAX_MISSING_SHARE` of the trimmed sample and are dropped; no protected control is. Of the 102 trimmed countries, 67 observe every remaining variable.

- **Countries lost to a single variable.** Seven countries are lost because of one variable: BDI, BWA, COG and LSO miss only `marketref`; DEU misses only `polity2`; SLE and ZMB miss only `capital_growth_pct_gdp_initial`.
- **Small states and territories.** KIR, SYC, VCT and PRI miss most of the geographic and institutional variables.

## 8. Kept versus dropped (descriptive)

Table 1 gives the status (kept or dropped) and the label (from `lbl_all`) of every variable and is written to `output/variable_status.tex`. Table 2 compares the countries in the final sample (kept) with all other countries in `dat` (dropped) for the outcome, the regressor and the Solow variables: the mean of each and the number of countries that observe it. No tests are computed.

In [ ]:
# 1. table 1: status and label of every variable
tab1 = pd.DataFrame({'status': ['dropped' if v in vv_dropped else 'kept' for v in dat.columns],
                     'label': [lbl_all.get(v, '') for v in dat.columns]}, index=dat.columns)

# 2. write table 1 to LaTeX
Path('output').mkdir(exist_ok=True)
Path('output/variable_status.tex').write_text(tabulate(tab1, headers='keys', tablefmt='latex_booktabs'))
tab1

In [ ]:
# 1. countries in the final sample
K = dat['code'].isin(dat_pre['code'])

# 2. table 2: means and counts for the outcome, the regressor and the Solow variables
vv = ['gdp_growth', 'lgdp_initial'] + list(vv_solow.values())
tab = pd.DataFrame({'mean kept': dat.loc[K, vv].mean(),
                    'mean dropped': dat.loc[~K, vv].mean(),
                    'n kept': dat.loc[K, vv].count(),
                    'n dropped': dat.loc[~K, vv].count()}).round(3)

# 3. write table 2 to LaTeX
Path('output').mkdir(exist_ok=True)
Path('output/kept_vs_dropped.tex').write_text(tabulate(tab, headers='keys', tablefmt='latex_booktabs'))
tab

Kept and dropped countries grew at almost the same rate (mean `gdp_growth` 0.016 against 0.015) and started from similar income levels (mean `lgdp_initial` 7.979 against 7.845), so the restrictions do not visibly select on the outcome or the regressor.

- **Similar.** Population growth is the same in both groups (0.017), and the 1970 investment rate differs little (21.306 against 22.272).
- **Different.** Kept countries have a lower secondary education share (`ls_bl` 12.982 against 17.642), a slightly lower tertiary share (`lh_bl` 3.18 against 3.557) and a lower average investment rate (`investment_rate` 22.523 against 24.38).
- **Coverage.** The dropped group is small for some variables, for example 30 countries for the 1970 investment rate, so those means rest on few countries.

This comparison is descriptive: it covers observed characteristics only and says nothing about differences in unobservables.

## 9. Summary

Final counts, the final control names in `vv_prescreened`, and the prescreened data `dat_pre` saved to `output/growth_prescreened.csv` for the estimation step.

In [ ]:
# 1. final counts
vv_prescreened = [v for v in dat_pre.columns if v not in ['code', 'gdp_growth', 'lgdp_initial']]
print(f'Countries (rows): {dat_pre.shape[0]}')
print(f'Explanatory variables (columns minus gdp_growth and code): {dat_pre.shape[1] - 2}')
print(f'Controls excluding lgdp_initial: {len(vv_prescreened)}')

# 2. save the prescreened data
dat_pre.to_csv('output/growth_postselection.csv', index=False)

# Simple OLS

In [ ]:
# 1. avoiding missings 
I = dat[['gdp_growth', 'lgdp_initial']].notnull().all(axis=1)

# 2. extract dataset 
y = dat.loc[I, 'gdp_growth'].values.reshape((-1,1)) * 100.0
X = dat.loc[I, ['constant','lgdp_initial']].values

# 3. run OLS
betahat = np.linalg.inv(X.T @ X) @ X.T @ y
print(betahat)

# Adding more controls

In [ ]:
vs = vv_all['geography'] + vv_all['religion']
xs = ['lgdp_initial', 'pop_growth', 'investment_rate'] + vs

# avoiding missings
all_vars = ['gdp_growth'] + xs
I = dat[all_vars].notnull().all(1)

# extract data
X = dat.loc[I, xs].values
y = dat.loc[I,'gdp_growth'].values.reshape((-1,1)) * 100. #easier to read output when growth is in 100%

# add const. (unless this breaks the rank condition)
oo = np.ones((I.sum(),1))
X = np.hstack([X, oo])
xs.append('constant') # we put it in as the last element

# check the rank condition
K = X.shape[1]
assert np.linalg.matrix_rank(X) == X.shape[1], f'X does not have full rank'

# compute the OLS estimator
betas = np.linalg.inv(X.T @ X) @ X.T @ y

In [ ]:
# format nicely
print(f'Mean y = {y.mean(): 5.2f}% growth per year')
pd.DataFrame({'β': betas[:,0]}, index=xs).round(3)